# NestedSimPy — a simple M/M/1 example

[NestedSimPy](https://nestedsimpy.github.io/) extends SimPy with *nested simulation*. This notebook walks through the same simple use case as the [Simple example](https://nestedsimpy.github.io/simple-example.html) docs page: we start with an **M/M/1 queue** in plain SimPy, then add nested simulation capabilities using NestedSimPy and inspect the outputs.

## 1. Install

_Pre-release: NestedSimPy installs from a hosted wheel. (After the public release this becomes `pip install nestedsimpy`.)_

In [ ]:
# Pre-release install from a hosted wheel (Google Drive).
!pip install -q gdown
import gdown
gdown.download(id="1N7mlgDVpVids6Ekr4p2e-gEUrUodiEuq",
               output="nestedsimpy-0.1.0-py3-none-any.whl", quiet=True)
!pip install -q "nestedsimpy-0.1.0-py3-none-any.whl[plot]"

import nestedsimpy
print("NestedSimPy ready —", len(nestedsimpy.__all__), "public objects")

## 2. An M/M/1 queue in SimPy

Here is a simple implementation of an M/M/1 queue using SimPy. It runs for 10 units of time.

In [ ]:
"""A plain M/M/1 queue in SimPy."""

import random

import simpy

ARRIVAL_RATE = 3.0  # mean arrivals per unit time
SERVICE_RATE = 4.0  # mean services per unit time
SIM_TIME = 10.0
SEED = 42


def customer(env, server):
    """Wait for the server, then hold it for an exponential service time."""
    with server.request() as request:
        yield request
        yield env.timeout(random.expovariate(SERVICE_RATE))


def arrivals(env, server):
    """Generate customers with exponential interarrival times."""
    while True:
        yield env.timeout(random.expovariate(ARRIVAL_RATE))
        env.process(customer(env, server))


random.seed(SEED)
env = simpy.Environment()
server = simpy.Resource(env, capacity=1)
env.process(arrivals(env, server))
env.run(until=SIM_TIME)

## 3. Nested simulation using NestedSimPy

Here is a modified version that runs the same M/M/1 queue using NestedSimPy. To convert the model we reuse the SimPy code, replacing certain SimPy objects with corresponding NestedSimPy objects — `simpy.Environment` becomes `nestedsimpy.NestedEnvironment`, `simpy.Resource` becomes `nestedsimpy.NestedResource`, and the `timeout` calls become `nested_timeout` with an explicit distribution. We then add a few commands that configure the execution of the nested simulation: at each customer **arrival** at the server (the *trigger event*) the outer simulation pauses and launches **3 independent inner branches** that each explore a possible future for 5 units of time (`set_inner_stopping_condition(relative_time=5.0)`), drawing their own random futures (`set_rng("independent")`). Outputs are stored under `nested_output/mm1`.

The model is written to a file and run as a subprocess; the output below is the **outer** trajectory and matches the plain SimPy example.

In [ ]:
%%writefile mm1_nested.py
"""The same M/M/1 queue under NestedSimPy.

At each arrival (the trigger event) the outer simulation pauses and launches
three independent inner branches that each explore a possible future from that
state.
"""

import nestedsimpy

ARRIVAL_RATE = 3.0  # mean arrivals per unit time
SERVICE_RATE = 4.0  # mean services per unit time
SIM_TIME = 10.0
SEED = 42


def customer(env, server):
    """Wait for the server, then hold it for an exponential service time."""
    with server.request() as request:
        yield request
        yield env.nested_timeout({"distribution": "exponential", "lambda": SERVICE_RATE})


def arrivals(env, server):
    """Generate customers with exponential interarrival times."""
    while True:
        yield env.nested_timeout({"distribution": "exponential", "lambda": ARRIVAL_RATE})
        env.process(customer(env, server))


env = nestedsimpy.NestedEnvironment()
server = nestedsimpy.NestedResource(env, capacity=1, nested_id="srv")
env.process(arrivals(env, server))
env.set_output_options(out_dir="nested_output/mm1", gzip_trace=False)
env.set_rng("independent")  # each branch draws its own future
env.set_outer_seed(SEED)
env.set_triggering_objects(nested_id="srv")
env.set_triggering_conditions({"on": "arrival", "frequency": 1})
env.set_inner_repetitions(3)
env.set_inner_stopping_condition(relative_time=5.0)
env.set_outer_stopping_condition(timeout=SIM_TIME)
env.nested_run()

In [ ]:
# Run as a subprocess so the outer output is clean (inner branches run in separate processes).
!python mm1_nested.py

## 4. NestedSimPy output

NestedSimPy offers visualization and data-generation functionalities. When the nested simulator runs, it stores intermediate outputs under `nested_output/mm1` (configured by the argument `out_dir` in the code above). Once the outer simulation terminates, we can create an `OutputManager` object initialized from that folder — even in a fresh session — to visualize and export the data.

In [ ]:
from nestedsimpy import OutputManager

# Load the completed nested run from its output folder
om = OutputManager("nested_output/mm1")
print(f"{len(om.triggers())} trigger events; the outer path has {len(om.export_outer_event_log())} recorded events")

### Visualization

`OutputManager` plots the run in four ways — the **outer** simulation on its own (static or interactive), and the **inner** simulations launched at a trigger event. All four show the **queue length** at the server over time. In the interactive outer plot, each dotted vertical line marks a trigger event; the inner plots show the grey outer context up to the trigger point followed by the inner branch future(s).

In [ ]:
from IPython.display import display

# Visualization: plot the outer simulation and the inner simulations
display(om.visualize_outer_static("outer.png"))      # display outer simulation
om.visualize_outer_interactive().show()              # display outer simulation
om.visualize_inner(trigger_id=0, inner_id=0).show()  # display a single inner simulation
om.visualize_inner(trigger_id=0).show()              # display all inner simulations

### Data

`OutputManager` also exports the run as tables of two kinds: **event logs** — one row per simulation event, giving a sample path of the run — and **case tables** — one row per case (here, per customer), giving its predicted outcomes. `export_outer_case_table()` produces the prediction table: for each triggering customer it reports the average outcome over its inner simulations — for example the mean waiting time, which estimates that customer's expected wait.

In [ ]:
# Data export: write the run out as CSV tables
om.export_inner_event_log(trigger_id=0, inner_id=0, path="inner.csv")   # export a single inner simulation
om.export_outer_event_log("outer.csv")                                  # export the outer simulation
om.export_outer_case_table("predictions.csv")   # export the outer simulation and aggregate the output of the inner simulations at every trigger event
print("wrote inner.csv, outer.csv, predictions.csv")